In [1]:
# Install required libraries
!pip install pandas pyarrow

import pandas as pd
import os

In [2]:
base_path = "/content/data_lake"

bronze_path = os.path.join(base_path, "bronze")
silver_path = os.path.join(base_path, "silver")
gold_path = os.path.join(base_path, "gold")

os.makedirs(bronze_path, exist_ok=True)
os.makedirs(silver_path, exist_ok=True)
os.makedirs(gold_path, exist_ok=True)

print("Data lake folders created!")

Data lake folders created!


In [3]:
# Example: upload file manually
from google.colab import files
uploaded = files.upload()

# Load dataset
df = pd.read_parquet(list(uploaded.keys())[0])

# Save raw data to Bronze layer
bronze_file = os.path.join(bronze_path, "taxi_raw.parquet")
df.to_parquet(bronze_file)

print("Raw data stored in Bronze layer")

Saving yellow_tripdata_2023-01.parquet to yellow_tripdata_2023-01.parquet
Raw data stored in Bronze layer


In [4]:
df = pd.read_parquet(bronze_file)

# Basic cleaning
df = df.dropna()

# Convert datetime columns
df['tpep_pickup_datetime'] = pd.to_datetime(df['tpep_pickup_datetime'])
df['tpep_dropoff_datetime'] = pd.to_datetime(df['tpep_dropoff_datetime'])

# Feature Engineering
df['trip_duration'] = (
    df['tpep_dropoff_datetime'] - df['tpep_pickup_datetime']
).dt.total_seconds() / 60  # minutes

# Filter invalid trips
df = df[df['trip_duration'] > 0]
df = df[df['trip_distance'] > 0]

# Save to Silver layer
silver_file = os.path.join(silver_path, "taxi_cleaned.parquet")
df.to_parquet(silver_file)

print("Cleaned data saved to Silver layer")

Cleaned data saved to Silver layer


In [5]:
df = pd.read_parquet(silver_file)

# Create analytics dataset
gold_df = df.groupby(df['tpep_pickup_datetime'].dt.date).agg({
    'trip_distance': 'mean',
    'fare_amount': 'mean',
    'total_amount': 'sum',
    'trip_duration': 'mean'
}).reset_index()

gold_df.columns = ['date', 'avg_distance', 'avg_fare', 'total_revenue', 'avg_duration']

# Save to Gold layer
gold_file = os.path.join(gold_path, "taxi_analytics.parquet")
gold_df.to_parquet(gold_file)

print("Gold dataset created!")

Gold dataset created!


In [6]:
def validate_data(df):
    assert df.isnull().sum().sum() == 0, "Missing values found!"
    assert (df['avg_distance'] >= 0).all(), "Invalid distance values!"
    assert (df['total_revenue'] >= 0).all(), "Invalid revenue values!"
    print("Data validation passed!")

validate_data(gold_df)

Data validation passed!


In [7]:
import logging

logging.basicConfig(level=logging.INFO)

logging.info("ETL Pipeline Started")
logging.info(f"Raw records: {len(df)}")
logging.info("Pipeline completed successfully!")

In [8]:
# Partition by year/month
df['year'] = df['tpep_pickup_datetime'].dt.year
df['month'] = df['tpep_pickup_datetime'].dt.month

partition_path = os.path.join(silver_path, "partitioned")

df.to_parquet(partition_path, partition_cols=['year', 'month'])

print("Partitioned data saved!")

Partitioned data saved!
